In [ ]:
#libraries used
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import FeatureHasher
import tensorflow as tf
from sklearn.svm import OneClassSVM
from sklearn.metrics import root_mean_squared_error,r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

#imported games.csv which is the steam games dataset 
gamesDf = pd.read_csv('games.csv')

#revenue for games with 0 games sold (estimated owners) will be 0 so removed them
gamesDf = gamesDf[gamesDf['Estimated owners'] != 0].reset_index(drop=True)

#revenue for games with a price of $0 will be 0 so removed them
gamesDf = gamesDf[gamesDf['Price'] != 0].reset_index(drop=True)

#target is the revenue which is calculated via [(games sold * price) * .7 ] need to multiply by .7 because steam takes 30% of revenue
targetDf = pd.DataFrame()
targetDf['revenue'] = gamesDf['Estimated owners'] * gamesDf['Price'] * .7

#relevant features for the revenue prediction
featuresDf = gamesDf[['Name', 'Release date', 'Estimated owners','Price', 'Score rank', 'Recommendations','Average playtime two weeks', 'Median playtime forever','Median playtime two weeks', 'Publishers', 'Categories','Genres', 'Tags']].copy()

#fills in NaN values for all features with 0 for numerical features 
featuresDf = featuresDf.fillna(0)

#fills in NaN values for all features with an empty string for non-numerical features 
nonNumericalFeat= [ 'Publishers', 'Categories', 'Genres', 'Tags','Release date']

#log transforms most of numerical features since the most popular games like Silent Hill will cause left skewed data then min max normalization for all
skewedFeat = ['Estimated owners','Score rank', 'Recommendations', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks']
featuresDf[skewedFeat] = np.log1p(featuresDf[skewedFeat])
numericalFeat = ['Estimated owners', 'Price', 'Score rank', 'Recommendations','Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks']
featuresDf[numericalFeat] = (featuresDf[numericalFeat] - featuresDf[numericalFeat].min()) / (featuresDf[numericalFeat].max() - featuresDf[numericalFeat].min())

#one hot encoding for non numerical features then join all
pubEncode = pd.get_dummies(featuresDf['Publishers'])
catEncode = pd.get_dummies(featuresDf['Categories'])
dateEncode = pd.get_dummies(featuresDf['Release date'])
genEncode = pd.get_dummies(featuresDf['Genres'])
tagEncode = pd.get_dummies(featuresDf['Tags'])
featurez = pd.concat([featuresDf[numericalFeat],pubEncode,catEncode,dateEncode,genEncode,tagEncode], axis=1)

#runs one class SVM to preprocess data further
x_array = featurez.to_numpy()
svm = OneClassSVM(kernel='rbf', gamma=0.001, nu=0.03)
svm.fit(x_array)
labels = svm.predict(x_array)
inlier = labels == 1
outlier = labels == -1
numOutliers = (labels == -1).sum()
numInliers = (labels == 1).sum()
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(16, 6))
#LHS graph shows original data
ax0.scatter(x_array[:, 0], x_array[:, 1], s=5, alpha=0.3, color='blue', label='All points')
ax0.set_title("Original Data")
ax0.set_xlabel("Estimated owners")
ax0.set_ylabel("Price")
ax0.legend(markerscale=3)

# RHS graph shows data after svm
ax1.scatter(x_array[inlier, 0], x_array[inlier, 1], s=5, alpha=0.3, color='blue', label='Inlier')
ax1.scatter(x_array[outlier, 0], x_array[outlier, 1], s=20, alpha=0.7, color='red', label='Outlier')
ax1.set_title("Detected Outliers using One Class SVM")
ax1.set_xlabel("Estimated owners")
ax1.set_ylabel("Price")
ax1.legend(markerscale=3)
plt.suptitle(f"One-Class SVM Outlier Detection | nu={svm.nu}, gamma={svm.gamma} | {numOutliers} outliers ({100*numOutliers/len(labels):.1f}%)")
plt.tight_layout()
plt.show()
print("Outliers:", outlier.sum(), "out of", len(labels))

#get rid of outliers in features & targets
featurez = featurez[inlier].reset_index(drop=True)
targetDf = targetDf[inlier].reset_index(drop=True)

#pairplot for relevant features
sns.pairplot(featuresDf[['Estimated owners', 'Price', 'Score rank', 'Recommendations','Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks']])
plt.show()

#split data into train:validation:test with 70:15:15 ratio had to log transform revenue due to extreme vals and model would not predict accurately w/out it 
x = featurez
y = np.log1p(targetDf['revenue'])
#split 70:30 train:temp
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, random_state = 66)
#split 15:15 validation:test
x_validation, x_test, y_validation, y_test = train_test_split(x_temp, y_temp, test_size=.5, random_state = 66)

#IMPORTANT: don't forget to reverse log transform y after validation/test completes e.g. y_test = np.expm1(y_test)
#IMPORTANT #2: USE TENSOR FLOW KERAS FOR MODEL ALL THE DATA WAS PRE PROCESSED ONLY FOR THIS also probably use like embeddings in tensorflow for non-numerical features cols (theres ALOT after one hot encoding) 

#MLP Add extra layers to allow non-linearity to the model
dimention = x_train.shape[1]

model = Sequential([
    Dense(256, activation='relu', input_shape=(dimention,)), #avoid vanishing gradient
    Dense(128, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='linear') #map indicators to single prediction
])

#Compile and Train
model.compile(optimizer='adam', loss='mse')

print("Training model")
history = model.fit(x_train, y_train, validation_data=(x_validation, y_validation),epochs=100, batch_size=2000,verbose=1) #2000 batch size to decerase time in fixing process

#Predictions and reverse log transform
y_pred_log = model.predict(x_test)
y_test_real = np.expm1(y_test) 
y_pred_real = np.expm1(y_pred_log).flatten()

print( "" )

#RMSE and R2
rmse = root_mean_squared_error(y_test_real, y_pred_real)
r2 = r2_score(y_test_real, y_pred_real)

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Training and Validation Loss (MSE)', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True)

plt.suptitle("Model Training Performance", fontsize=16)
plt.tight_layout()
plt.show()

MemoryError: Unable to allocate 1.04 GiB for an array with shape (11438, 12201) and data type object